# GaiaLab Naija Assistant — QLoRA notebook

This notebook prepares an explicitly licensed dataset and, only when you run the final training cell, fine-tunes a LoRA adapter. The 100 bundled rows are original drafts pending Nigerian human review, not a production corpus. **No training results are claimed.** Use a Python 3.11 GPU runtime and review all data and model licences first.

## 1. Runtime and repository setup

In Colab choose Runtime → Change runtime type → GPU. In Kaggle enable a GPU in notebook settings. Clone the repository if it is not already present.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Run this only from the directory containing requirements.txt.
%pip install -q -r requirements.txt

## 2. Validate provenance and create splits

The validator fails on missing fields, missing source/licence metadata, and empty instructions or outputs. Read the report and inspect the actual rows before continuing.

In [ ]:
!python -m src.validate_dataset data/gaialab_naija_v0.1.jsonl --output-dir prepared_data
!python -m src.prepare_dataset --train prepared_data/train.jsonl --validation prepared_data/validation.jsonl --output-dir prepared_data/hf

In [ ]:
import json
from pathlib import Path

rows = [json.loads(line) for line in Path('data/gaialab_naija_v0.1.jsonl').read_text(encoding='utf-8').splitlines()]
assert len(rows) == 100, 'GaiaLab Naija Dataset v0.1 should contain exactly 100 rows.'
print('Review these sources/licences before training:', sorted({(r['source'], r['license']) for r in rows})[:3], '...')
print('Languages:', sorted({r['language'] for r in rows}))
print('Categories:', sorted({r['category'] for r in rows}))

## 3. Configure an explicit QLoRA run

Complete Nigerian human review and expand the draft with sufficiently varied, consented, and licensed data before meaningful training. The default model is small enough for many free GPU sessions, but availability and memory vary. Running the next cell starts training and may take time.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = 'outputs/gaialab-naija-adapter'
print('Model:', MODEL_ID)
print('Adapter output:', OUTPUT_DIR)

In [ ]:
# EXPLICIT TRAINING CELL — do not run until data and licences have been reviewed.
!python -m src.train_qlora --model-id {MODEL_ID} --dataset-dir prepared_data/hf --output-dir {OUTPUT_DIR} --epochs 1

## 4. Review and optional publishing

Download the adapter first. Evaluate it with `evaluation/evaluate_model.py`, document failures, and create a complete model card. If publishing, store `HF_TOKEN` in Colab/Kaggle secrets, start with a private Hugging Face repository, and push only after checking base-model and dataset licences. Never place a token in this notebook.